In [6]:
# Cell 3 — Khởi động API server nhẹ chạy trên Colab
# Thay vLLM bằng FastAPI + transformers để tránh lỗi cài đặt CUDA / dependency
import threading
import time
import torch
from fastapi import FastAPI
from pydantic import BaseModel
from transformers import AutoModelForCausalLM, AutoTokenizer
import uvicorn

MODEL_NAME = "TinyLlama/TinyLlama-1.1B-Chat-v1.0"

print(f"Loading model: {MODEL_NAME}")
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    torch_dtype=torch.float16 if torch.cuda.is_available() else torch.float32,
    device_map="auto" if torch.cuda.is_available() else None,
)

app = FastAPI()

class ChatRequest(BaseModel):
    model: str | None = None
    messages: list[dict]
    max_tokens: int = 128
    temperature: float = 0.7

@app.get("/health")
def health():
    return {"status": "ok"}

@app.post("/v1/chat/completions")
def chat_completions(req: ChatRequest):
    prompt = tokenizer.apply_chat_template(req.messages, tokenize=False, add_generation_prompt=True)
    inputs = tokenizer(prompt, return_tensors="pt")
    if torch.cuda.is_available():
        inputs = {k: v.to(model.device) for k, v in inputs.items()}
    output_ids = model.generate(
        **inputs,
        max_new_tokens=req.max_tokens,
        do_sample=req.temperature > 0,
        temperature=req.temperature,
    )
    generated = tokenizer.decode(output_ids[0][inputs["input_ids"].shape[-1]:], skip_special_tokens=True)
    return {
        "id": "chatcmpl-colab",
        "object": "chat.completion",
        "choices": [
            {"index": 0, "message": {"role": "assistant", "content": generated}, "finish_reason": "stop"}
        ],
    }

def run_server():
    uvicorn.run(app, host="0.0.0.0", port=8001)

threading.Thread(target=run_server, daemon=True).start()
print("Colab chat server starting in background on port 8001.")
time.sleep(20)
print("Nếu model chưa tải xong, hãy tăng thời gian chờ lên 40-60 giây.")


Loading model: TinyLlama/TinyLlama-1.1B-Chat-v1.0


config.json:   0%|          | 0.00/608 [00:00<?, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

tokenizer.model:   0%|          | 0.00/500k [00:00<?, ?B/s]

...

Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/124 [00:00<?, ?B/s]

Colab chat server starting in background on port 8001.


INFO:     Started server process [565]
INFO:     Waiting for application startup.
INFO:     Application startup complete.
INFO:     Uvicorn running on http://0.0.0.0:8001 (Press CTRL+C to quit)


Nếu model chưa tải xong, hãy tăng thời gian chờ lên 40-60 giây.


# Google Colab vLLM-Style Setup (ngrok)

Notebook này được chỉnh cho Google Colab để tránh xung đột dependency khi cài gói. Thay vì cài vLLM nặng, notebook dùng một API server nhẹ với `transformers` để phục vụ endpoint tương thích `/v1/chat/completions`, rồi expose qua ngrok.


## Yêu cầu trước khi bắt đầu

- Google Colab với GPU đã bật: Runtime → Change runtime type → Hardware accelerator = GPU.
- Kết nối Internet trên Colab.
- Tài khoản ngrok (https://ngrok.com) để lấy `authtoken`.


In [1]:
# Cell 1 — Cài dependencies cho Google Colab
# Dùng cài đặt tối thiểu để tránh xung đột với các gói đã có sẵn trong Colab
%pip install -q --upgrade pip
%pip install -q jedi fastapi uvicorn pyngrok sentence-transformers transformers accelerate bitsandbytes

import torch
print('CUDA available:', torch.cuda.is_available())
print('Torch version:', torch.__version__)


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.8/1.8 MB 26.8 MB/s eta 0:00:00
CUDA available: True
Torch version: 2.10.0+cu128


## Lấy ngrok authtoken (ngrok.com)

1. Truy cập https://ngrok.com và đăng ký / đăng nhập.
2. Mở Dashboard → Auth & API keys → copy `Authtoken`.
3. Trên máy local bạn có thể lưu token bằng CLI: `ngrok config add-authtoken <YOUR_TOKEN>`.
4. Trên Kaggle Notebook, bạn có thể thiết lập token tạm thời bằng Python (dưới).

Lưu ý: không chia sẻ authtoken công khai.

In [2]:
# Cell 2 — Thiết lập ngrok token trong session (thay YOUR_NGROK_TOKEN)
from pyngrok import ngrok
# Thay YOUR_NGROK_TOKEN bằng token lấy từ ngrok.com
ngrok.set_auth_token("3DseDGjT8WjGEBSxxcv8O9tWds9_5UaoVgkRDGutsGmYniEnW")


## Tạo tunnel ngrok cho vLLM và in public URL

Sử dụng `pyngrok` để mở tunnel HTTP tới port 8001 và in ra public URL — copy URL này để dán vào `.env` local.

In [3]:
# Cell 4 — Tạo ngrok tunnel cho vLLM
from pyngrok import ngrok

# Mở tunnel HTTP cho port 8001
vllm_tunnel = ngrok.connect(8001, "http")
vllm_url = vllm_tunnel.public_url
print("vLLM URL (copy this to local .env):", vllm_url)

vLLM URL (copy this to local .env): https://gigabyte-font-broadband.ngrok-free.dev


## Khởi động embedding API server (FastAPI + sentence-transformers)

Chạy một FastAPI nhỏ để trả embedding; service này chạy trên port 8002 và cũng sẽ được expose bằng ngrok.

In [4]:
# Cell 5 — Embedding API server (runs in background)
from fastapi import FastAPI
from sentence_transformers import SentenceTransformer
import uvicorn, threading, time

app = FastAPI()
# Nếu bị thiếu RAM/GPU, đổi sang model nhẹ hơn: 'all-MiniLM-L6-v2'
model = SentenceTransformer("BAAI/bge-small-en-v1.5")

@app.post("/embed")
def embed(data: dict):
    texts = data.get("texts", [])
    embeddings = model.encode(texts).tolist()
    return {"embeddings": embeddings}

def run_embed():
    uvicorn.run(app, host="0.0.0.0", port=8002)

threading.Thread(target=run_embed, daemon=True).start()
print("Embedding server started on port 8002 (background).")
time.sleep(3)


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:93: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/124 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/52.0 [00:00<?, ?B/s]

...

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/125 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

INFO:     Started server process [565]
INFO:     Waiting for application startup.
INFO:     Application startup complete.
INFO:     Uvicorn running on http://0.0.0.0:8002 (Press CTRL+C to quit)


Embedding server started on port 8002 (background).


In [5]:
# Cell 6 — Tạo ngrok tunnel cho embedding service
from pyngrok import ngrok
embed_tunnel = ngrok.connect(8002, "http")
embed_url = embed_tunnel.public_url
print("Embedding URL (copy this to local .env):", embed_url)

Embedding URL (copy this to local .env): https://gigabyte-font-broadband.ngrok-free.dev


## Cập nhật file `.env` trên local

Copy cả hai URL in ra ở trên vào file `.env` trên máy local (thư mục dự án):

Example `.env`:
```
VLLM_NGROK_URL=PASTE_VLLM_URL_HERE
EMBED_NGROK_URL=PASTE_EMBED_URL_HERE
LANGCHAIN_API_KEY=your_langsmith_key
LANGCHAIN_PROJECT=lab28-platform
```

Sau đó khởi động lại local Docker stack nếu cần (`docker compose up -d`).

## Kiểm tra kết nối từ máy local (ví dụ)

Trên máy local, bạn có thể kiểm tra endpoint vLLM và embedding bằng `curl` hoặc `requests`. Ví dụ (local):

In [ ]:
# Cell 7 — Ví dụ kiểm tra (chạy trên máy local, không phải Kaggle)
import requests
import os
vllm_url = os.environ.get("VLLM_NGROK_URL")  # hoặc paste trực tiếp URL
embed_url = os.environ.get("EMBED_NGROK_URL")

# Test embedding service
if embed_url:
    r = requests.post(f"{embed_url}/embed", json={"texts": ["hello world"]}, timeout=30)
    print("embed status:", r.status_code, r.json())

# Test vLLM chat completions (API may differ by vLLM version)
if vllm_url:
    payload = {"model": "Qwen/Qwen2.5-7B-Instruct-GPTQ-Int4", "messages": [{"role": "user", "content": "Say hi"}]}
    r2 = requests.post(f"{vllm_url}/v1/chat/completions", json=payload, timeout=60)
    print("vllm status:", r2.status_code)
    try:
        print(r2.json())
    except Exception:
        print("vLLM response not JSON or failed")

## Troubleshooting nhanh
- Nếu model load quá lâu: tăng `time.sleep()` sau khi khởi động vLLM hoặc kiểm tra logs trên Kaggle kernel.
- Nếu ngrok không mở được tunnel: kiểm tra authtoken và giới hạn tài khoản (free tier giới hạn tunnels).
- Nếu embedding model quá nặng: chọn model nhỏ hơn (ví dụ `all-MiniLM-L6-v2`) để tiết kiệm RAM/GPU.
- Luôn copy URL private (ngrok) vào `.env` trên local và giữ bí mật.